In [ ]:
"""
ArabPhon Rule-Based Phoneme Parser
Input : a fully diacritised Arabic word (with tashkil)
Output: IPA-style phoneme sequence as a list, e.g. ['k','a','t','a','b','a']

Handles:
  - Basic consonants + short vowels (fatha/kasra/damma)
  - Sukun (closed syllable, no vowel after consonant)
  - Shadda (gemination — consonant doubled)
  - Tanwin (fathatan/kasratan/dammatan → /an/, /in/, /un/)
  - Long vowels (medd): alef after fatha, ya after kasra, waw after damma
  - Sun-letter assimilation of ال (al-)
  - Hamza variants (all map to /ʔ/)
  - Ta marbuta (ة) → /a/ in pause form (grade 1 default)
  - Alef maqsura (ى) → /aː/
"""

# Cell 1 — Mount Drive

In [ ]:

# Mount Google Drive to access the Tashkeela dataset stored in /Arabphon/TASHKEELA_DIR
from google.colab import drive
drive.mount('/content/drive')

TASHKEELA_DIR = '/content/drive/MyDrive/Arabphon/TASHKEELA_DIR'



# Cell 2 — Rule-based phoneme parser

In [ ]:
"""
ArabPhon Rule-Based Phoneme Parser
Input : a fully diacritised Arabic word (with tashkil)
Output: IPA-style phoneme sequence as a list, e.g. ['k','a','t','a','b','a']

Handles:
  - Basic consonants + short vowels (fatha/kasra/damma)
  - Sukun (closed syllable, no vowel after consonant)
  - Shadda (gemination — consonant doubled)
  - Tanwin (fathatan/kasratan/dammatan → /an/, /in/, /un/)
  - Long vowels (medd): alef after fatha, ya after kasra, waw after damma
  - Sun-letter assimilation of ال (al-)
  - Hamza variants (all map to /ʔ/)
  - Ta marbuta (ة) → /a/ in pause form (grade 1 default)
  - Alef maqsura (ى) → /aː/
"""

# ── Unicode constants ──────────────────────────────────────────────────────────
FATHA     = 'َ'  # َ
KASRA     = 'ِ'  # ِ
DAMMA     = 'ُ'  # ُ
SUKUN     = 'ْ'  # ْ
SHADDA    = 'ّ'  # ّ
FATHATAN  = 'ً'  # ً
KASRATAN  = 'ٍ'  # ٍ
DAMMATAN  = 'ٌ'  # ٌ
TATWEEL   = 'ـ'  # ـ (kashida — ignore)

DIACRITICS = {FATHA, KASRA, DAMMA, SUKUN, SHADDA,
              FATHATAN, KASRATAN, DAMMATAN}

# ── Consonant map ──────────────────────────────────────────────────────────────
CONSONANTS = {
    'ب': 'b',  'ت': 't',  'ث': 'θ',  'ج': 'dʒ', 'ح': 'ħ',
    'خ': 'x',  'د': 'd',  'ذ': 'ð',  'ر': 'r',  'ز': 'z',
    'س': 's',  'ش': 'ʃ',  'ص': 'sˤ', 'ض': 'dˤ', 'ط': 'tˤ',
    'ظ': 'ðˤ', 'ع': 'ʕ',  'غ': 'ɣ',  'ف': 'f',  'ق': 'q',
    'ك': 'k',  'ل': 'l',  'م': 'm',  'ن': 'n',  'ه': 'h',
    'و': 'w',  'ي': 'j',
    # Hamza variants → glottal stop
    'ء': 'ʔ',  'أ': 'ʔ',  'إ': 'ʔ',  'آ': 'ʔ',  'ؤ': 'ʔ',  'ئ': 'ʔ',
    # Special
    'ة': 'h',   # ta marbuta — overridden in pause form below
    'ى': None,  # alef maqsura — handled as long vowel
    'ا': None,  # alef — handled as long vowel carrier
}

SUN_LETTERS = set('تثدذرزسشصضطظلن')

# ── Darija difficulty flags ────────────────────────────────────────────────────
DARIJA_ABSENT = {'θ', 'ð', 'ðˤ', 'sˤ', 'dˤ', 'q', 'ʔ'}


def parse_word(word: str, pause_form: bool = True) -> dict:
    """
    Parse a fully diacritised Arabic word into a phoneme sequence.

    Returns:
        {
          'word': original word,
          'phonemes': ['k','a','t','a','b','a'],
          'blending_script': 'k-a-t-a-b-a',
          'darija_flags': ['θ','ðˤ', ...],  # phonemes absent from Darija
          'difficulty': 'low' | 'medium' | 'high'
        }
    """
    # Remove kashida
    word = word.replace(TATWEEL, '')

    chars = list(word)
    phonemes = []
    i = 0

    # Handle definite article ال with sun-letter assimilation
    if len(chars) >= 2 and chars[0] == 'ا' and chars[1] == 'ل':
        # Find the next consonant after ال
        j = 2
        while j < len(chars) and chars[j] in DIACRITICS:
            j += 1
        if j < len(chars) and chars[j] in SUN_LETTERS:
            # Sun letter: ال → assimilate, e.g. الشَّمْس → aʃ-ʃams
            phonemes.append('a')
            # skip ا and ل, let the sun letter be doubled via shadda logic
            i = 2
        else:
            # Moon letter: ال → /al/
            phonemes.extend(['a', 'l'])
            i = 2

    while i < len(chars):
        ch = chars[i]

        # Skip standalone diacritics (already consumed with their consonant)
        if ch in DIACRITICS:
            i += 1
            continue

        # Look ahead for diacritics attached to this character
        diacritics_ahead = []
        j = i + 1
        while j < len(chars) and chars[j] in DIACRITICS:
            diacritics_ahead.append(chars[j])
            j += 1

        has_shadda   = SHADDA    in diacritics_ahead
        has_sukun    = SUKUN     in diacritics_ahead
        has_fatha    = FATHA     in diacritics_ahead
        has_kasra    = KASRA     in diacritics_ahead
        has_damma    = DAMMA     in diacritics_ahead
        has_fathatan = FATHATAN  in diacritics_ahead
        has_kasratan = KASRATAN  in diacritics_ahead
        has_dammatan = DAMMATAN  in diacritics_ahead

        # ── Alef madda (آ = ʔ + aː always) ──
        if ch == 'آ':
            phonemes.extend(['ʔ', 'aː'])
            i = j
            continue

        # ── Alef (long vowel carrier) ──
        # Only reached when alef was NOT already consumed by the fatha look-ahead above.
        # This happens for word-initial bare alef (undiacritised) — skip silently.
        if ch == 'ا':
            # Word-initial alef with kasra or damma = hamzat al-wasl → emit vowel
            if i == 0 or (len(phonemes) == 0):
                if has_kasra:
                    phonemes.append('i')
                elif has_damma:
                    phonemes.append('u')
                elif has_fatha:
                    phonemes.append('a')
            # else: mid-word bare alef = long vowel carrier (already handled by previous consonant's fatha→aː)
            i = j
            continue
        # ── Alef maqsura ──
        if ch == 'ى':
            phonemes.append('aː')
            i = j
            continue

        # ── Ta marbuta ──
        if ch == 'ة':
            if pause_form:
                # In pause form, ة is silent (the preceding vowel already covers it)
                pass
            else:
                phonemes.append('t')
                if has_fathatan:   phonemes.extend(['a', 'n'])
                elif has_kasratan: phonemes.extend(['i', 'n'])
                elif has_dammatan: phonemes.extend(['u', 'n'])
                elif has_fatha:    phonemes.append('a')
                elif has_kasra:    phonemes.append('i')
                elif has_damma:    phonemes.append('u')
            i = j
            continue  # <-- skips the vowel block below entirely

        # ── Regular consonant ──
        consonant_ipa = CONSONANTS.get(ch)
        if consonant_ipa is None:
            # Unknown character — skip
            i = j
            continue

        # Shadda = gemination (consonant said twice)
        if has_shadda:
            phonemes.append(consonant_ipa)
            phonemes.append(consonant_ipa)
        else:
            phonemes.append(consonant_ipa)

        # Implied vowels for hamza carriers with no explicit diacritic
        no_explicit_vowel = not (has_fatha or has_kasra or has_damma or has_sukun or
                                 has_fathatan or has_kasratan or has_dammatan)
        if no_explicit_vowel:
            if ch == 'أ':
                phonemes.append('a')
            elif ch in ('إ', 'ئ'):
                phonemes.append('i')
            elif ch == 'ؤ':
                phonemes.append('u')
            # ء, آ → no implied vowel (آ already handled above)

        # Vowel after consonant
        if has_fathatan:
            phonemes.extend(['a', 'n'])
            # Consume the orthographic alef that follows fathatan (spelling convention, not a vowel)
            if j < len(chars) and chars[j] == 'ا':
                j += 1
        elif has_kasratan:
            phonemes.extend(['i', 'n'])
        elif has_dammatan:
            phonemes.extend(['u', 'n'])
        elif has_fatha:
            # Check next non-diacritic char for long vowel (medd)
            # Only bare alef (ا) and alef maqsura (ى) extend to /aː/
            # Hamza-bearing alef variants are NOT medd — they are the next consonant
            next_char = chars[j] if j < len(chars) else ''
            if next_char == 'ا':
                phonemes.append('aː')
                j += 1  # consume the alef
            elif next_char == 'ى':
                phonemes.append('aː')
                j += 1
            else:
                phonemes.append('a')
        elif has_kasra:
            next_char = chars[j] if j < len(chars) else ''
            # Only treat ي as long-vowel extension if it carries NO diacritics of its own
            if next_char == 'ي':
                k = j + 1
                next_ya_diac = []
                while k < len(chars) and chars[k] in DIACRITICS:
                    next_ya_diac.append(chars[k])
                    k += 1
                if not next_ya_diac:
                    # bare ي = long /iː/
                    phonemes.append('iː')
                    j += 1
                else:
                    # ي has its own diacritics → short /i/, leave ي to be processed next
                    phonemes.append('i')
            else:
                phonemes.append('i')
        elif has_damma:
            next_char = chars[j] if j < len(chars) else ''
            if next_char == 'و':
                phonemes.append('uː')
                j += 1
                # Consume orthographic alef after waw (e.g. بَرَعُوا — the ا is silent)
                if j < len(chars) and chars[j] == 'ا':
                    j += 1
            else:
                phonemes.append('u')
        elif has_sukun:
            pass  # no vowel — closed syllable
        # else: no diacritic — undiacritised character, skip vowel

        i = j

    # ── Darija difficulty ──
    flags = [p for p in phonemes if p in DARIJA_ABSENT]
    n = len(flags)
    difficulty = 'low' if n == 0 else ('medium' if n <= 2 else 'high')

    return {
        'word': word,
        'phonemes': phonemes,
        'blending_script': '-'.join(phonemes),
        'darija_flags': flags,
        'difficulty': difficulty,
    }


# ── Batch processing ───────────────────────────────────────────────────────────
def parse_text(text: str) -> list:
    """Parse all words in a text string."""
    results = []
    for word in text.split():
        word = word.strip('،.,؟!()[]""')
        if word:
            results.append(parse_word(word))
    return results


# ── Demo ───────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    test_words = [
        'كَتَبَ',      # kataba
        'الشَّمْسُ',   # aʃ-ʃamsu (sun letter assimilation + shadda + sukun)
        'مَدْرَسَةٌ',  # madrasa (tanwin dammatan)
        'ثَعْلَبٌ',    # θaʕlabun (difficult: θ and ʕ absent from Darija)
        'بِسْمِ',      # bismi (sukun)
        'كِتَابٌ',     # kitaːbun (long vowel)
    ]

    print(f"{'Word':<20} {'Phonemes':<35} {'Difficulty':<10} {'Darija flags'}")
    print('-' * 80)
    for w in test_words:
        r = parse_word(w)
        print(f"{r['word']:<20} {r['blending_script']:<35} {r['difficulty']:<10} {r['darija_flags']}")

# Cell 3 — Silver label generation

In [ ]:
# Silver Label Generation from Tashkeela Corpus
# Applies the rule-based parser to all diacritised words in selected Tashkeela folders
# (كتب حديثة, aljazeera, manual) — chosen for modern MSA vocabulary
# Output: silver_labels.csv with columns [word, phonemes, difficulty]
# These labels are called "silver" because they are machine-generated, not human-annotated

import os, csv

FOLDERS = ['كتب حديثة', 'aljazeera', 'manual']
OUTPUT_CSV = 'silver_labels.csv'


pairs = []
skipped = 0

for folder in FOLDERS:
    path = os.path.join(TASHKEELA_DIR, folder)
    if not os.path.exists(path):
        print(f"Folder not found: {path}")
        continue
    for root, _, files in os.walk(path):
        for fname in files:
            fpath = os.path.join(root, fname)
            try:
                text = open(fpath, encoding='utf-8').read()
            except Exception:
                skipped += 1
                continue
            for word in text.split():
                word = word.strip('،.,؟!()[]""«»‏‎')
                if len(word) < 2:
                    continue
                result = parse_word(word)
                if result['phonemes']:
                    pairs.append((word, result['blending_script'], result['difficulty']))

print(f"Total pairs: {len(pairs):,}")
print(f"Files skipped (encoding errors): {skipped}")

with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['word', 'phonemes', 'difficulty'])
    writer.writerows(pairs)

print(f"Saved to {OUTPUT_CSV}")


import re
import unicodedata

bad_chars = set('{}[]():؟،.')
DIACRITICS_SET = set('ًٌٍَُِّْ')

def is_clean(word):
    # No punctuation/bracket garbage
    if any(c in bad_chars for c in word): return False
    # No spaces of any kind (multi-word instances)
    if any(unicodedata.category(c) == 'Zs' for c in word): return False
    if re.search(r'\s', word): return False
    # Too short
    if len(word) < 2: return False
    # Must have at least one diacritic
    if not any(c in DIACRITICS_SET for c in word): return False
    return True

pairs = [(w, p, d) for w, p, d in pairs if is_clean(w)]
print(f"After cleaning: {len(pairs)} rows")

OUTPUT_CSV = '/content/drive/MyDrive/Arabphon/silver_labels.csv'

with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['word', 'phonemes', 'difficulty'])
    writer.writerows(pairs)

print(f"Saved to {OUTPUT_CSV}")

# Cell 4a — Deduplication and train/val/test split

In [ ]:
# Deduplication and Dataset Split
# Removes duplicate word entries (same diacritised form appearing in multiple files)
# Splits unique pairs into 80% train / 10% validation / 10% test
# Output: train.csv, val.csv, test.csv

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Arabphon/silver_labels.csv')

print(f"Before dedup: {len(df):,}")

df = df.drop_duplicates(subset='word')
print(f"After dedup:  {len(df):,}")

# Split 80/10/10
train = df.sample(frac=0.8, random_state=42)
remaining = df.drop(train.index)
val = remaining.sample(frac=0.5, random_state=42)
test = remaining.drop(val.index)

train.to_csv('/content/drive/MyDrive/Arabphon/train.csv', index=False)
val.to_csv('/content/drive/MyDrive/Arabphon/val.csv', index=False)
test.to_csv('/content/drive/MyDrive/Arabphon/test.csv', index=False)

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

# Cell 4b — Dataset Statistics

In [ ]:
import pandas as pd
train = pd.read_csv('/content/drive/MyDrive/Arabphon/train.csv')
val   = pd.read_csv('/content/drive/MyDrive/Arabphon/val.csv')
test  = pd.read_csv('/content/drive/MyDrive/Arabphon/test.csv')

all_df = pd.concat([train, val, test])
print(f"Src vocab (unique chars): {all_df['word'].apply(list).explode().nunique()}")
print(f"Tgt vocab (unique phonemes): {all_df['phonemes'].str.split('-').explode().nunique()}")
print(f"Avg word length: {all_df['word'].apply(len).mean():.1f} chars")
print(f"Avg phoneme seq length: {all_df['phonemes'].str.split('-').apply(len).mean():.1f}")
print(f"\nDifficulty dist (train):\n{train['difficulty'].value_counts()}")

# Cell 4c - Darija difficulty distribution

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/Arabphon/train.csv')
all_df = pd.concat([
    pd.read_csv('/content/drive/MyDrive/Arabphon/train.csv'),
    pd.read_csv('/content/drive/MyDrive/Arabphon/val.csv'),
    pd.read_csv('/content/drive/MyDrive/Arabphon/test.csv')
])

counts = all_df['difficulty'].value_counts().reindex(['low', 'medium', 'high'])
colors = ['#2ecc71', '#f39c12', '#e74c3c']

plt.figure(figsize=(6, 4))
bars = plt.bar(counts.index, counts.values, color=colors, edgecolor='black')
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{val:,}\n({val/len(all_df)*100:.1f}%)',
             ha='center', fontsize=10)
plt.title('Darija Difficulty Distribution', fontsize=13)
plt.xlabel('Difficulty Level')
plt.ylabel('Number of Words')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Arabphon/difficulty_distribution.png', dpi=150)
plt.show()

# ── Cell 5a: Character-level Seq2Seq Model (model_a - Experiment 1 - 0 noise) ──

In [ ]:
# Trains a character-level LSTM encoder-decoder on silver labels
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from collections import Counter

# ── 1. Build vocabularies ──
train_df = pd.read_csv('/content/drive/MyDrive/Arabphon/train.csv')
val_df   = pd.read_csv('/content/drive/MyDrive/Arabphon/val.csv')

PAD, SOS, EOS, UNK = '<PAD>', '<SOS>', '<EOS>', '<UNK>'

def build_vocab(sequences):
    tokens = [t for seq in sequences for t in seq]
    counts = Counter(tokens)
    vocab = [PAD, SOS, EOS, UNK] + sorted(counts.keys())
    return {t: i for i, t in enumerate(vocab)}, vocab

def build_phoneme_vocab(sequences):
    tokens = [t for seq in sequences for t in seq]
    counts = Counter(tokens)
    vocab = [PAD, SOS, EOS, UNK] + sorted(counts.keys())
    return {t: i for i, t in enumerate(vocab)}, vocab

src_seqs = [list(w) for w in pd.concat([train_df['word'], val_df['word']])]
tgt_seqs = [p.split('-') for p in pd.concat([train_df['phonemes'], val_df['phonemes']])]

src_vocab, src_tokens = build_vocab(src_seqs)
tgt_vocab, tgt_tokens = build_phoneme_vocab(tgt_seqs)
tgt_inv = {i: t for t, i in tgt_vocab.items()}

print(f"Src vocab: {len(src_vocab)} | Tgt vocab: {len(tgt_vocab)}")

# ── 2. Dataset ──
class PhonemeDataset(Dataset):
    def __init__(self, df, src_vocab, tgt_vocab, max_len=40):
        self.data = []
        for _, row in df.iterrows():
            src = [src_vocab.get(c, src_vocab[UNK]) for c in list(row['word'])]
            tgt = [tgt_vocab[SOS]] + \
                  [tgt_vocab.get(p, tgt_vocab[UNK]) for p in row['phonemes'].split('-')] + \
                  [tgt_vocab[EOS]]
            self.data.append((src[:max_len], tgt[:max_len]))

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

def collate(batch):
    src_batch, tgt_batch = zip(*batch)
    max_src = max(len(s) for s in src_batch)
    max_tgt = max(len(t) for t in tgt_batch)
    src_pad = torch.tensor([s + [0]*(max_src-len(s)) for s in src_batch])
    tgt_pad = torch.tensor([t + [0]*(max_tgt-len(t)) for t in tgt_batch])
    return src_pad, tgt_pad

train_loader = DataLoader(PhonemeDataset(train_df, src_vocab, tgt_vocab),
                          batch_size=128, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(PhonemeDataset(val_df, src_vocab, tgt_vocab),
                          batch_size=128, collate_fn=collate)

# ── 3. Model ──
class Encoder(nn.Module):
    def __init__(self, vocab, emb, hid, layers, drop):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.rnn = nn.LSTM(emb, hid, layers, batch_first=True,
                           dropout=drop, bidirectional=True)
        self.fc  = nn.Linear(hid*2, hid)

    def forward(self, x):
        out, (h, c) = self.rnn(self.emb(x))
        h = torch.tanh(self.fc(torch.cat([h[-2], h[-1]], dim=1)))
        h = h.unsqueeze(0).repeat(LAYERS, 1, 1)
        c = torch.zeros_like(h)
        return out, h, c

class Decoder(nn.Module):
    def __init__(self, vocab, emb, hid, layers, drop):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.rnn = nn.LSTM(emb, hid, layers, batch_first=True, dropout=drop)
        self.fc  = nn.Linear(hid, vocab)

    def forward(self, x, h, c):
        out, (h, c) = self.rnn(self.emb(x.unsqueeze(1)), (h, c))
        return self.fc(out.squeeze(1)), h, c

class Seq2Seq(nn.Module):
    def __init__(self, enc, dec, tgt_vocab_size):
        super().__init__()
        self.enc = enc; self.dec = dec
        self.tgt_vocab_size = tgt_vocab_size

    def forward(self, src, tgt, teacher_forcing=0.5):
        batch, tgt_len = tgt.shape
        outputs = torch.zeros(batch, tgt_len, self.tgt_vocab_size).to(src.device)
        _, h, c = self.enc(src)
        inp = tgt[:, 0]
        for t in range(1, tgt_len):
            out, h, c = self.dec(inp, h, c)
            outputs[:, t] = out
            inp = tgt[:, t] if torch.rand(1) < teacher_forcing else out.argmax(1)
        return outputs

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

EMB, HID, LAYERS, DROP = 128, 256, 2, 0.5
enc = Encoder(len(src_vocab), EMB, HID, LAYERS, DROP)
dec = Decoder(len(tgt_vocab), EMB, HID, LAYERS, DROP)
model = Seq2Seq(enc, dec, len(tgt_vocab)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# ── 4. Training ──
EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        out = model(src, tgt)
        loss = criterion(out[:, 1:].reshape(-1, len(tgt_vocab)), tgt[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ── 5. Save ──
torch.save(model.state_dict(), '/content/drive/MyDrive/Arabphon/lstm_model.pt')
torch.save({'src_vocab': src_vocab, 'tgt_vocab': tgt_vocab,
            'tgt_inv': tgt_inv}, '/content/drive/MyDrive/Arabphon/vocabs.pt')
print("Model saved.")

# ── Cell 6a: LSTM Evaluation — Phoneme Error Rate (PER) ──


In [ ]:
# ── Cell 6a: LSTM Evaluation — Clean Model on Test Set ──
import pandas as pd
from editdistance import eval as edit_distance

UNK = '<UNK>'; SOS = '<SOS>'; EOS = '<EOS>'

torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
model.eval()
print(model.training)  # must print False

def predict(word, mdl, max_len=40):
    src = [src_vocab.get(c, src_vocab[UNK]) for c in list(word)]
    src_t = torch.tensor([src]).to(device)
    with torch.no_grad():
        _, h, c = mdl.enc(src_t)
        inp = torch.tensor([tgt_vocab[SOS]]).to(device)
        result = []
        for _ in range(max_len):
            out, h, c = mdl.dec(inp, h, c)
            pred = out.argmax(1).item()
            token = tgt_inv[pred]
            if token == EOS:
                break
            result.append(token)
            inp = torch.tensor([pred]).to(device)
    return result

test_df = pd.read_csv('/content/drive/MyDrive/Arabphon/test.csv')

total_errors, total_phones, exact_matches = 0, 0, 0
for _, row in test_df.iterrows():
    ref  = row['phonemes'].split('-')
    pred = predict(row['word'], model)
    total_errors += edit_distance(pred, ref)
    total_phones  += len(ref)
    exact_matches += int(pred == ref)

print(f"Test samples : {len(test_df):,}")
print(f"Exact match  : {exact_matches/len(test_df)*100:.1f}%")
print(f"PER          : {total_errors/total_phones*100:.2f}%")

# Cell 7 — Per-Difficulty Evaluation (Experiment 1, Clean Labels)

In [ ]:
from collections import defaultdict
from editdistance import eval as edit_distance

stats_a = defaultdict(lambda: {'exact': 0, 'n': 0, 'errors': 0, 'phones': 0})
for _, row in test_df.iterrows():
    pred = predict(row['word'], model)
    ref  = row['phonemes'].split('-')
    diff = row['difficulty']
    stats_a[diff]['exact']  += int(pred == ref)
    stats_a[diff]['n']      += 1
    stats_a[diff]['errors'] += edit_distance(pred, ref)
    stats_a[diff]['phones'] += len(ref)

print(f"{'Difficulty':<12} {'N':>6} {'Exact%':>8} {'PER%':>8}")
print('-'*38)
for d in ['low', 'medium', 'high']:
    s = stats_a[d]
    print(f"{d:<12} {s['n']:>6} {s['exact']/s['n']*100:>7.1f}% {s['errors']/s['phones']*100:>7.2f}%")

# CELL 8. Ablation: Dropout Rate ─  (model_a - Experiment 1)

In [ ]:
# ── Cell . Ablation: Dropout Rate ──
DROPOUT_VALS = [0.5]
ABL_EPOCHS   = 20

abl_results = []

torch.manual_seed(42)
torch.backends.cudnn.deterministic = True

abl_val_loader = DataLoader(
    PhonemeDataset(val_df, src_vocab, tgt_vocab),
    batch_size=128, collate_fn=collate
)

def greedy_decode_abl(model, src_ids):
    model.eval()
    with torch.no_grad():
        src_t = torch.tensor([src_ids], device=device)
        _, h, c = model.enc(src_t)
        inp = torch.tensor([tgt_vocab[SOS]], device=device)
        preds = []
        for _ in range(40):
            out, h, c = model.dec(inp, h, c)
            tok = out.argmax(1).item()
            if tok == tgt_vocab[EOS]:
                break
            preds.append(tok)
            inp = torch.tensor([tok], device=device)
    return preds

def eval_abl(model):
    exact, per_total, per_ref = 0, 0, 0
    for src_ids, tgt_ids in abl_val_loader.dataset.data:
        pred = greedy_decode_abl(model, src_ids)
        ref  = [t for t in tgt_ids[1:] if t not in (tgt_vocab[EOS], 0)]
        exact += int(pred == ref)
        m, n = len(ref), len(pred)
        dp = list(range(n+1))
        for r in ref:
            ndp = [dp[0]+1]
            for j, p in enumerate(pred):
                ndp.append(min(dp[j+1]+1, ndp[-1]+1, dp[j]+(r!=p)))
            dp = ndp
        per_total += dp[n]; per_ref += max(m,1)
    N = len(abl_val_loader.dataset)
    return exact/N*100, per_total/per_ref*100

for drop in DROPOUT_VALS:
    print(f"\nDropout={drop}")
    enc_abl   = Encoder(len(src_vocab), EMB, HID, LAYERS, drop)
    dec_abl   = Decoder(len(tgt_vocab), EMB, HID, LAYERS, drop)
    model_abl = Seq2Seq(enc_abl, dec_abl, len(tgt_vocab)).to(device)
    opt_abl   = torch.optim.Adam(model_abl.parameters(), lr=1e-3)
    crit_abl  = nn.CrossEntropyLoss(ignore_index=0)

    abl_train_loader = DataLoader(
        PhonemeDataset(train_df, src_vocab, tgt_vocab),
        batch_size=128, shuffle=True, collate_fn=collate
    )

    for epoch in range(ABL_EPOCHS):
        model_abl.train()
        total_loss = 0
        for src, tgt in abl_train_loader:
            src, tgt = src.to(device), tgt.to(device)
            opt_abl.zero_grad()
            out = model_abl(src, tgt)
            loss = crit_abl(out[:,1:].reshape(-1, len(tgt_vocab)), tgt[:,1:].reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_abl.parameters(), 1.0)
            opt_abl.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1:02d} | Loss {total_loss/len(abl_train_loader):.4f}")

    exact_pct, per_pct = eval_abl(model_abl)
    abl_results.append({'dropout': drop, 'exact': exact_pct, 'per': per_pct})
    print(f"  → Exact: {exact_pct:.1f}% | PER: {per_pct:.2f}%")

print("\n── Ablation Summary ──")
print(f"{'Dropout':<10}{'Exact %':<12}{'PER %'}")
print('-'*30)
for r in abl_results:
    print(f"{r['dropout']:<10}{r['exact']:<12.1f}{r['per']:.2f}")

# ── Cell 9: Experiment 2 — Noisy Data Preparation ─────────────────────

In [ ]:
# Creates noisy training set with 30% diacritic dropout
# Shadda excluded (gemination marker — dropping it contradicts the label)
# Val/test sets remain clean (realistic OCR/input noise scenario)
# Output: train_noisy.csv → used by Cell 5b (LSTM) and Cell 7b (AraBERT)

import random, pandas as pd

# Droppable diacritics — excludes shadda (U+0651)
DROPPABLE = set('\u064e\u064f\u0650\u0652\u064b\u064c\u064d')
# fatha, damma, kasra, sukun, fathatan, dammatan, kasratan

def add_noise(word: str, drop_prob: float = 0.3, rng=None) -> str:
    if rng is None:
        rng = random
    return ''.join(
        '' if c in DROPPABLE and rng.random() < drop_prob else c
        for c in word
    )

# ── Load splits ──
train_df = pd.read_csv('/content/drive/MyDrive/Arabphon/train.csv')
val_df   = pd.read_csv('/content/drive/MyDrive/Arabphon/val.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/Arabphon/test.csv')

# Apply noise — reproducible RNG
rng = random.Random(42)
train_noisy = train_df.copy()
train_noisy['word'] = train_noisy['word'].apply(lambda w: add_noise(w, rng=rng))

print(f"Train (noisy): {len(train_noisy):,}")
print(f"Val   (clean): {len(val_df):,}")
print(f"Test  (clean): {len(test_df):,}")

# Verify noise
print("\nNoise check (original → noisy):")
for i in [0, 1, 2, 3, 4]:
    print(f"  {train_df['word'].iloc[i]}  →  {train_noisy['word'].iloc[i]}")

# Save
train_noisy.to_csv('/content/drive/MyDrive/Arabphon/train_noisy.csv', index=False)
print("\nSaved train_noisy.csv — ready for Cell 5b and Cell 7b")

# ── Cell 10 (5b): LSTM seq2seq — Experiment 2 (Noisy Training) ──────

In [ ]:
# Cell 5b — LSTM Noisy Training (Experiment 2)
# Trains on train_noisy.csv, evaluates on clean test set

import torch
import torch.nn as nn
import pandas as pd

# ── Hyperparameters ──
EMB, HID, LAYERS, DROP = 128, 256, 2, 0.5
EPOCHS, BATCH = 20, 128

# ── Load data ──
train_noisy_df = pd.read_csv('/content/drive/MyDrive/Arabphon/train_noisy.csv')
val_df_b       = pd.read_csv('/content/drive/MyDrive/Arabphon/val.csv')

# ── Build vocab from noisy train + clean val ──
src_seqs_b = [list(w) for w in pd.concat([train_noisy_df['word'], val_df_b['word']])]
tgt_seqs_b = [p.split('-') for p in pd.concat([train_noisy_df['phonemes'], val_df_b['phonemes']])]

src_vocab_b, src_tokens_b = build_vocab(src_seqs_b)
tgt_vocab_b, tgt_tokens_b = build_phoneme_vocab(tgt_seqs_b)
tgt_inv_b = {i: t for t, i in tgt_vocab_b.items()}

print(f"Src vocab: {len(src_vocab_b)} | Tgt vocab: {len(tgt_vocab_b)}")

# ── Dataloaders ──
train_loader_b = DataLoader(
    PhonemeDataset(train_noisy_df, src_vocab_b, tgt_vocab_b),
    batch_size=BATCH, shuffle=True, collate_fn=collate
)
val_loader_b = DataLoader(
    PhonemeDataset(val_df_b, src_vocab_b, tgt_vocab_b),
    batch_size=BATCH, collate_fn=collate
)

# ── Model ──
enc_b   = Encoder(len(src_vocab_b), EMB, HID, LAYERS, DROP)
dec_b   = Decoder(len(tgt_vocab_b), EMB, HID, LAYERS, DROP)
model_b = Seq2Seq(enc_b, dec_b, len(tgt_vocab_b)).to(device)

optimizer_b = torch.optim.Adam(model_b.parameters(), lr=1e-3)
criterion_b = nn.CrossEntropyLoss(ignore_index=0)

# ── Training ──
for epoch in range(EPOCHS):
    model_b.train()
    total_loss = 0
    for src, tgt in train_loader_b:
        src, tgt = src.to(device), tgt.to(device)
        optimizer_b.zero_grad()
        out = model_b(src, tgt)
        loss = criterion_b(out[:, 1:].reshape(-1, len(tgt_vocab_b)), tgt[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_b.parameters(), 1.0)
        optimizer_b.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader_b):.4f}")

# ── Save ──
torch.save(model_b.state_dict(), '/content/drive/MyDrive/Arabphon/lstm_model_noisy.pt')
torch.save({'src_vocab': src_vocab_b, 'tgt_vocab': tgt_vocab_b,
            'tgt_inv': tgt_inv_b}, '/content/drive/MyDrive/Arabphon/vocabs_noisy.pt')
print("Model saved.")

# ── Cell 11 (6b): LSTM Evaluation — Experiment 2 (Noisy Training) ───────────

In [ ]:
# Cell 6b — Evaluate Noisy LSTM (model_b) on Clean Test Set

import pandas as pd
from editdistance import eval as edit_distance

PAD, SOS, EOS, UNK = '<PAD>', '<SOS>', '<EOS>', '<UNK>'

test_df = pd.read_csv('/content/drive/MyDrive/Arabphon/test.csv')


torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
model_b.eval()
print(model_b.training)  # must print False

def predict_b(word, max_len=40):
    src = [src_vocab_b.get(c, src_vocab_b[UNK]) for c in list(word)]
    src_t = torch.tensor([src]).to(device)
    with torch.no_grad():
        _, h, c = model_b.enc(src_t)
        inp = torch.tensor([tgt_vocab_b[SOS]]).to(device)
        result = []
        for _ in range(max_len):
            out, h, c = model_b.dec(inp, h, c)
            pred = out.argmax(1).item()
            token = tgt_inv_b[pred]
            if token == EOS:
                break
            result.append(token)
            inp = torch.tensor([pred]).to(device)
    return result

total_errors, total_phones, exact_matches = 0, 0, 0
for _, row in test_df.iterrows():
    ref  = row['phonemes'].split('-')
    pred = predict_b(row['word'])
    total_errors  += edit_distance(pred, ref)
    total_phones  += len(ref)
    exact_matches += int(pred == ref)

n = len(test_df)
print(f"LSTM Noisy — Experiment 2")
print(f"Test samples : {n:,}")
print(f"Exact match  : {exact_matches/n*100:.1f}%")
print(f"PER          : {total_errors/total_phones*100:.2f}%")

# Cell 12 - Qualitative examples  (model_b - Experiment 2)

In [ ]:
# Cell — Qualitative Examples

greedy_decode, encode = make_decoder(model_b, src_vocab_b, tgt_vocab_b, tgt_inv_b)

correct, wrong = [], []

for _, row in test_df.iterrows():
    src = encode(row['word'])          # ← assign to src
    pred = greedy_decode(src)
    ref  = row['phonemes'].split('-')
    if pred == ref:
        correct.append({'word': row['word'], 'phonemes': row['phonemes']})
    else:
        wrong.append({'word': row['word'], 'reference': row['phonemes'],
                      'predicted': '-'.join(pred)})

correct_df = pd.DataFrame(correct)
wrong_df   = pd.DataFrame(wrong)

print(f"Correct: {len(correct_df)} / Wrong: {len(wrong_df)}")

print("\nWith shadda (ّ):")
print(correct_df[correct_df['word'].str.contains('ّ')].head(3).to_string())

print("\nWith sun-letter ال:")
print(correct_df[correct_df['word'].str.startswith('الش') | correct_df['word'].str.startswith('الر')].head(3).to_string())

print("\nWith tanwin:")
print(correct_df[correct_df['phonemes'].str.endswith('-n')].head(3).to_string())

print("\nWith long vowel aː:")
print(correct_df[correct_df['phonemes'].str.contains('aː')].head(3).to_string())

print("\n=== WRONG PREDICTIONS (sample) ===")
print(wrong_df.head(10).to_string())

# Cell 13 — Robustness to Missing Diacritics (model_b - Experiment 2)

In [ ]:
# Cell — Robustness: no-diacritic input (parser vs LSTM)

greedy_decode, encode = make_decoder(model_b, src_vocab_b, tgt_vocab_b, tgt_inv_b)

sample = test_df.sample(50, random_state=42).reset_index(drop=True)

DIACRITICS_SET = set('ًٌٍَُِّْ')
def strip_diacritics(word):
    return ''.join(c for c in word if c not in DIACRITICS_SET)

print(f"{'Word':<20} {'Stripped':<12} {'Gold':<30} {'Parser (no diac)':<30} {'LSTM (no diac)'}")
print('-' * 130)
for _, row in sample.head(10).iterrows():
    gold = row['phonemes']
    stripped = strip_diacritics(row['word'])

    # Parser on stripped word
    try:
        parser_out = parse_word(stripped)
        if isinstance(parser_out, dict):
            parser_out = parser_out.get('phonemes', str(list(parser_out.keys())))
    except Exception as e:
        parser_out = f"ERR:{e}"

    # LSTM on stripped word
    src = encode(stripped)
    lstm_out = '-'.join(greedy_decode(src))

    print(f"{row['word']:<20} {stripped:<12} {gold:<30} {str(parser_out):<30} {lstm_out}")

# Cell 14— Learning Curve


In [ ]:
# Cell — Learning Curve

import matplotlib.pyplot as plt

losses_a = [2.1620, 1.1495, 0.4732, 0.1697, 0.0914, 0.0606, 0.0441, 0.0373,
            0.0306, 0.0283, 0.0250, 0.0224, 0.0207, 0.0182, 0.0179, 0.0165,
            0.0154, 0.0150, 0.0133, 0.0124]

losses_b = [2.2102, 1.4244, 0.7458, 0.4210, 0.3021, 0.2423, 0.2081, 0.1872,
            0.1718, 0.1600, 0.1507, 0.1430, 0.1318, 0.1272, 0.1216, 0.1199,
            0.1114, 0.1093, 0.1041, 0.1009]

epochs = range(1, 21)

plt.figure(figsize=(8, 4))
plt.plot(epochs, losses_a, label='Exp 1: Clean (DROP=0.5)', marker='o', markersize=4)
plt.plot(epochs, losses_b, label='Exp 2: Noisy (DROP=0.5)', marker='s', markersize=4)
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss Curves — ArabPhon LSTM')
plt.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Arabphon/loss_curve.png', dpi=150)
plt.show()
print("Saved to Drive.")

# Cell 15 — Per-Difficulty Evaluation (model_b - Experiment 2)


In [ ]:
# Cell 15 — Per-Difficulty Evaluation (model_b)

import pandas as pd
from editdistance import eval as edit_distance

PAD, SOS, EOS, UNK = '<PAD>', '<SOS>', '<EOS>', '<UNK>'
test_df = pd.read_csv('/content/drive/MyDrive/Arabphon/test.csv')

def predict_b(word, max_len=40):
    src = [src_vocab_b.get(c, src_vocab_b[UNK]) for c in list(word)]
    src_t = torch.tensor([src]).to(device)
    with torch.no_grad():
        _, h, c = model_b.enc(src_t)
        inp = torch.tensor([tgt_vocab_b[SOS]]).to(device)
        result = []
        for _ in range(max_len):
            out, h, c = model_b.dec(inp, h, c)
            pred = out.argmax(1).item()
            token = tgt_inv_b[pred]
            if token == EOS:
                break
            result.append(token)
            inp = torch.tensor([pred]).to(device)
    return result

from collections import defaultdict
stats = defaultdict(lambda: {'errors': 0, 'phones': 0, 'exact': 0, 'n': 0})

for _, row in test_df.iterrows():
    ref  = row['phonemes'].split('-')
    pred = predict_b(row['word'])
    diff = row['difficulty']
    stats[diff]['errors'] += edit_distance(pred, ref)
    stats[diff]['phones'] += len(ref)
    stats[diff]['exact']  += int(pred == ref)
    stats[diff]['n']      += 1
    stats['overall']['errors'] += edit_distance(pred, ref)
    stats['overall']['phones'] += len(ref)
    stats['overall']['exact']  += int(pred == ref)
    stats['overall']['n']      += 1

print(f"{'Difficulty':<12} {'N':>6} {'Exact%':>8} {'PER%':>8}")
print('-'*38)
for diff in ['low', 'medium', 'high', 'overall']:
    s = stats[diff]
    print(f"{diff:<12} {s['n']:>6} {s['exact']/s['n']*100:>7.1f}% {s['errors']/s['phones']*100:>7.2f}%")